In [1]:
!pip uninstall -y pyarrow datasets
!pip install -U pyarrow datasets

Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Successfully uninstalled datasets-4.8.5
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (48.9 MB)
Using cached datasets-4.8.5-py3-none-any.whl (528 kB)


In [2]:
import pandas as pd

train_url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"

data = pd.read_csv(train_url, sep="\t")

print(data.head())
print("전체 데이터 개수:", len(data))

         id                                           document  label
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                                  너무재밓었다그래서보는것을추천한다      0
3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1
전체 데이터 개수: 150000


In [3]:
data = data.dropna(subset=["document", "label"])

data = data[["document", "label"]]
data = data.rename(columns={"document": "text"})

data["label"] = data["label"].astype(int)

data = data.sample(n=5000, random_state=42).reset_index(drop=True)

print(data.head())
print("사용할 데이터 개수:", len(data))
print(data["label"].value_counts())

                         text  label
0                      원본이 최고      1
1            스릴감과 훈훈함이 있는 영화.      1
2      굉장히 저평가되는 영화중 하나라고 생각함      1
3  정말영화같은이야기 영화여서 영화같은이야기가 좋다      1
4                 계기도없는데 이상하다      0
사용할 데이터 개수: 5000
label
0    2540
1    2460
Name: count, dtype: int64


In [4]:
!pip -q install -U transformers

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "klue/bert-base"

id2label = {
    0: "부정",
    1: "긍정"
}

label2id = {
    "부정": 0,
    "긍정": 1
}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

print("Tokenizer와 Model 불러오기 완료")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer와 Model 불러오기 완료


In [6]:
!pip -q install -U datasets

In [7]:
from datasets import Dataset

dataset = Dataset.from_pandas(data)

dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = dataset["train"]
valid_dataset = dataset["test"]

print(train_dataset)
print(valid_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 4000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})


In [8]:
MAX_LENGTH = 64

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

In [9]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_valid = valid_dataset.map(tokenize_function, batched=True)

print(tokenized_train)
print(tokenized_valid)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4000
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1000
})


In [10]:
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_valid = tokenized_valid.remove_columns(["text"])

tokenized_train.set_format("torch")
tokenized_valid.set_format("torch")

print(tokenized_train)
print(tokenized_valid)

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4000
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1000
})


In [11]:
import numpy as np
import torch

from sklearn.metrics import accuracy_score
from transformers import TrainingArguments, Trainer

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy}

In [13]:
training_args = TrainingArguments(
    output_dir="./week4_results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.355190,0.387883,0.832000


TrainOutput(global_step=250, training_loss=0.39602510070800784, metrics={'train_runtime': 2775.4412, 'train_samples_per_second': 1.441, 'train_steps_per_second': 0.09, 'total_flos': 131555527680000.0, 'train_loss': 0.39602510070800784, 'epoch': 1.0})

In [16]:
eval_result = trainer.evaluate()

print(eval_result)
print("Validation Accuracy:", round(eval_result["eval_accuracy"], 4))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.355190,0.387883,1,0.832000


{'eval_loss': 0.3878825604915619, 'eval_accuracy': 0.832}
Validation Accuracy: 0.832


In [17]:
import torch
from transformers import pipeline

sentiment_model = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_sentences = [
    "이 영화 진짜 재미있어요!",
    "완전 지루하고 시간 아까웠다.",
    "배우 연기는 좋았지만 스토리가 조금 아쉬웠다."
]

for sentence in test_sentences:
    result = sentiment_model(sentence)[0]

    print("입력 문장:", sentence)
    print("예측 결과:", result["label"])
    print("확신도:", round(result["score"], 4))
    print("-" * 50)

입력 문장: 이 영화 진짜 재미있어요!
예측 결과: 긍정
확신도: 0.9874
--------------------------------------------------
입력 문장: 완전 지루하고 시간 아까웠다.
예측 결과: 부정
확신도: 0.9647
--------------------------------------------------
입력 문장: 배우 연기는 좋았지만 스토리가 조금 아쉬웠다.
예측 결과: 부정
확신도: 0.8436
--------------------------------------------------
